# 08 Valuation

**What this does.** The check the method is built for: estimate each target's EV/EBITDA from its comps (median of
the comps' multiples) and compare it with the actual multiple. "No matching" (median of every training firm) is the
reference; intervals are a bootstrap over targets. EV/EBITDA is the answer key, so only this component and
`10_final_holdout` read `financials.csv`. These numbers check chosen lists; they are never used to choose a list,
a facet or a prompt.

All targets here are **training** targets (the ones the method was developed on). The holdout valuation is in 10.

**Changelog**
- 2026-11-19: created from `v1/scripts/valuation.py` and `v1/notebooks/comps_and_valuation.ipynb`. Valuation code
  that was in `agent_ranking.py` / `agent_loop_eval.py` moved here (`agent_valuation.py`). Values are for the
  current lists (Sonnet + Fiona sub-sector B3, Sonnet selector); v1's numbers were for the Haiku B3 list and are
  not reproduced here (decision 2026-11-19).
- 2026-11-19: v1 section 7's measures (`vl.summary`) and its showcase per-target table added for the current
  rules. v1's other sections stay in v1: v0 routes (z >= 2 candidates, Haiku LLM routes), the holdout sections,
  company size (section 9, the record of dropping F9), and the 11-10 integration study (sections 13-23). The
  fall-off charts moved to 03 and the facet section (8) to 04.


In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "v1a" / "current.toml").exists())
sys.path.insert(0, str(ROOT / "v1a" / "common"))
import paths, nbtools as nb
paths.setup()
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 90); pd.set_option("display.max_rows", 80)
cur = paths.current()
checks = nb.Checks()

In [2]:
import valuation as vl, agent_valuation as av
pd.Series(cur["valuation"])

financials    outputs-local/v1a/08_valuation/financials.csv
estimate             median EV/EBITDA of the selected peers
dtype: object

## 2. Inputs

In [3]:
nb.inputs([cur["valuation"]["financials"], cur["retrieval"]["candidates"] + "/candidates.csv", cur["selection"]["judgments"]])

,input,rows,sha256,status
0,outputs-local/v1a/08_valuation/financials.csv,560,a173be1a3a04,matches seed
1,outputs-local/v1a/03_retrieval/selector_candidates/candidates.csv,3730,1544b96bfc27,matches seed
2,outputs-local/v1a/04_selection/b3_selector_candidates_2026-11-04a_claude-cli_claude-so...,2538,dec35ac48af2,new in v1a


In [4]:
m = av.multiples()
training = av.training_pool().tolist()
actual = m[training]
top = pd.read_csv(paths.path(cur["retrieval"]["candidates"]) / "candidates.csv", dtype={"target": str, "comp": str})
b3 = top[top["selected"]]
j = pd.read_csv(paths.path(cur["selection"]["judgments"]), dtype={"target": str, "comp": str})
kept = b3.merge(j.loc[j["score"] == 1, ["target", "comp"]], on=["target", "comp"])
print(f"{len(training)} training targets; B3 {len(b3)} pairs; selector kept {len(kept)} pairs on {kept.target.nunique()} targets")

373 training targets; B3 2538 pairs; selector kept 507 pairs on 265 targets


## 3. Results

### 3.1 B3 and the selector on the 373 training targets

`selector, B3 if none kept`: a target where the selector keeps no comp falls back to its B3 list.

In [5]:
fallback = pd.concat([kept, b3[~b3["target"].isin(kept["target"])]])
results = {"no matching": vl.no_matching(training, training, m),
           "B3 (median)": vl.estimate(b3, m, how="median"),
           "B3 (similarity-weighted mean)": vl.estimate(b3, m, weight="similarity"),
           "selector, B3 if none kept (median)": vl.estimate(fallback, m, how="median")}
vl.bootstrap(results, actual, "no matching").round(1)

,targets,median abs % error,error 2.5%,error 97.5%,gain on baseline,gain 2.5%,gain 97.5%
comp rule,,,,,,,
no matching,373,31.9,29.4,34.4,NaN,NaN,NaN
B3 (median),373,33.8,30.3,37.5,-1.8,-5.3,1.6
B3 (similarity-weighted mean),373,30.2,27.7,35.1,1.7,-2.0,4.2
"selector, B3 if none kept (median)",373,31.3,28.2,34.4,0.6,-2.7,3.6


**v1's step-7 measures** (`comps_and_valuation.ipynb` section 7): median absolute % error, % of targets within
25%, Spearman (estimate, actual) and % of targets closer than with no matching, for the same rules.

In [6]:
evaluation = vl.summary(results, actual, baseline="no matching")
evaluation.round(2)


,targets,comps per target (median),median abs % error,% within 25%,"Spearman(estimate, actual)",% closer than no matching
comp rule,,,,,,
no matching,373,372.0,31.92,38.87,NaN,NaN
B3 (median),373,7.0,33.75,38.87,0.25,50.40
B3 (similarity-weighted mean),373,7.0,30.21,40.75,0.29,51.21
"selector, B3 if none kept (median)",373,2.0,31.29,40.21,0.38,49.60


**The ten showcase targets** (v1 section 7's per-target table, on the current rules): actual multiple, and each
rule's estimate and % error.

In [7]:
SHOWCASE = ["PCTY", "WEN", "CSL", "AFG", "BOC", "NWSA", "AZZ", "BBCP", "BUKS", "SNFC.A"]   # business-structure pilot targets
info = pd.read_csv(paths.path(cur["universe"]["universe"]), dtype=str).set_index("IQ_COMPANY_ID")
per_target = pd.concat({name: vl.errors(r, actual) for name, r in results.items()},
                       names=["comp rule", "IQ_COMPANY_ID"]).reset_index()
per_target.insert(2, "ticker", per_target["IQ_COMPANY_ID"].map(info["target_ciq_ticker"]))
show = per_target[per_target["ticker"].isin(SHOWCASE)]
SHOW_RULES = list(results)
wide = show.pivot(index="ticker", columns="comp rule", values=["estimate", "error %"])
wide = wide.reindex(SHOWCASE).reindex(columns=SHOW_RULES, level=1)
wide.insert(0, ("actual", ""), actual.rename(index=info["target_ciq_ticker"]).reindex(SHOWCASE))
wide.round(1)


actual    estimate                                                                                  error %              \
comp rule        no matching B3 (median) B3 (similarity-weighted mean) selector, B3 if none kept (median) no matching B3 (median)   
ticker                                                                                                                              
PCTY        22.1        10.6        13.4                          15.2                               17.1       -52.1       -39.4   
WEN         10.2        10.6        14.4                          13.9                               28.3         3.9        40.5   
CSL         12.0        10.6        19.4                          15.4                                5.4       -11.8        60.9   
AFG          9.3        10.6        10.1                          11.0                               10.1        14.0         8.0   
BOC         12.1        10.6         8.6                           8.8                                8.8       -12.2       -29.2   
NWSA        12.7        10.6         8.3                           8.1                                8.5       -16.5       -34.4   
AZZ          6.4        10.6        10.2                          11.6                               10.2        66.8        60.1   
BBCP         6.9        10.6        12.4                          13.1                               12.4        53.0        78.5   
BUKS         7.3        10.6         9.7                          10.0                                9.7        45.2        32.0   
SNFC.A       5.7        10.6         9.6                           9.8                                9.7        87.5        70.1   

                                                                            
comp rule B3 (similarity-weighted mean) selector, B3 if none kept (median)  
ticker                                                                      
PCTY                              -31.5                              -22.9  
WEN                                35.6                              176.3  
CSL                                27.8                              -55.0  
AFG                                18.4                                8.0  
BOC                               -27.4                              -27.4  
NWSA                              -36.0                              -33.1  
AZZ                                81.6                               60.1  
BBCP                               88.6                               78.5  
BUKS                               36.3                               32.0  
SNFC.A                             71.9                               70.2

**Reading.**
- **Measures:** every rule is closer than no matching for about half the targets (49.6-51.2%), and within 25% for
  38.9-40.8% (no matching 38.9%). Rank agreement with the actual multiple is modest: Spearman 0.25 (B3 median), 0.29
  (similarity-weighted) and 0.38 (selector with fallback).
- **Showcase targets:**
  - **Low-multiple targets are overestimated by every rule.** AZZ, BBCP, BUKS and SNFC.A trade at 5.7-7.3x; no
    matching gives 10.6x and the comp rules 9.6-13.1x.
  - **PCTY (22.1x) is underestimated by every rule;** the selector's three direct comps (ASUR, PAYC, ADP) come closest (17.1x, -23%).
  - **One-comp lists swing.** The selector keeps only BROS for WEN (28.3x against 10.2x, +176%) and only OC for CSL
    (5.4x against 12.0x, -55%).


### 3.2 Selector vs B3 on the same targets

Only targets where the selector kept at least one comp, so both rules value the same companies.

In [8]:
common = sorted(kept["target"].unique())
same = {"no matching": vl.no_matching(common, training, m),
        "B3 (median)": vl.estimate(b3[b3.target.isin(common)], m, how="median"),
        "selector kept (median)": vl.estimate(kept, m, how="median")}
vl.bootstrap(same, m[common], "no matching").round(1)

,targets,median abs % error,error 2.5%,error 97.5%,gain on baseline,gain 2.5%,gain 97.5%
comp rule,,,,,,,
no matching,265,31.5,27.8,34.1,NaN,NaN,NaN
B3 (median),265,32.6,28.3,37.0,-1.1,-5.5,2.9
selector kept (median),265,30.6,25.5,33.2,0.9,-3.1,5.9


### 3.3 Agent-ranking orderings on the 45 reviewed targets

Top 3 and top 6 of each ordering from `05_agent_grading` (pending Michael's review of the ranking).

In [9]:
import agent_ranking as ar
v = pd.read_csv(paths.path(cur["agent_grading"]["run"]) / "verdicts.csv", dtype={"target": str, "comp": str})
rv = av.ranking_valuation(ar.rank(v[v["status"] == "ok"]))
rv

,targets,median abs % error,error 2.5%,error 97.5%,gain on baseline,gain 2.5%,gain 97.5%
comp rule,,,,,,,
final top 3,45,35.9,22.5,56.0,8.9,-9.4,24.3
E top 3,45,42.4,23.4,58.3,2.5,-13.6,23.3
B3 top 3,45,42.4,23.5,49.5,2.5,-8.4,21.4
agent 0/1/2 top 3,45,44.0,24.2,49.6,0.9,-10.0,20.5
final top 6,45,31.5,21.8,49.5,13.4,-7.8,24.8
E top 6,45,31.5,21.8,48.9,13.4,-4.8,24.1
B3 top 6,45,32.4,24.0,49.5,12.5,-7.7,24.2
agent 0/1/2 top 6,45,31.5,24.0,48.9,13.4,-7.2,24.2
agent >= 1 (all kept),41,31.0,24.4,45.1,10.9,-4.0,19.0


### 3.4 GPT judge as a filter (saved, 11-10)

The GPT-6.1 Sol grades (07) used as a filter of Barbara's 293 reviewed B3 pairs, 45 targets. Saved table, moved here
from 07.

In [10]:
G = paths.out("08_valuation") / "gpt61sol_293_selection"
pd.read_csv(G / "selection_valuation.csv").round(1)

,comp rule,targets,comps per target (median),median abs % error,% within 25%,"Spearman(estimate, actual)",% closer than no matching
0,Reviewed B3 pairs,45,7.0,31.8,40.0,0.2,46.7
1,Judge score >= 1,44,5.0,31.0,38.6,0.3,47.7
2,Judge score = 2,10,1.0,19.5,60.0,0.0,50.0
3,No matching,45,372.0,44.9,28.9,NaN,NaN


## 4. Checks

In [11]:
checks.add("every valued target is a training target", set(results["B3 (median)"].index) <= set(training)
           and set(rv.index) and set(v["target"]) <= set(training))
checks.add("the B3 list is the current one (2,538 pairs)", len(b3) == 2538 and b3["target"].nunique() == 373)
checks.add("selector judgments cover the B3 list", len(b3.merge(j, on=["target", "comp"])) == len(b3))
checks.add("every training firm has a multiple", actual.notna().all(), f"{actual.notna().sum()}/{len(actual)}")
fin_cols = [c for c in pd.read_csv(paths.path(cur["valuation"]["financials"]), nrows=1).columns if "ebitda" in c.lower()]
reads = [p.relative_to(ROOT).as_posix() for p in (ROOT / "v1a").glob("0[2-7]_*/scripts/**/*.py")
         if any(w in p.read_text(encoding="utf-8") for w in ["financials.csv", '["financials"]', *fin_cols])]
checks.add("no 02-07 script reads the financials (01 only filters the 3-30 range)", not reads, ", ".join(reads))
checks.show()

5/5 checks pass


,check,result,detail
0,every valued target is a training target,pass,
1,"the B3 list is the current one (2,538 pairs)",pass,
2,selector judgments cover the B3 list,pass,
3,every training firm has a multiple,pass,373/373
4,no 02-07 script reads the financials (01 only filters the 3-30 range),pass,


## 5. History (v1)

- **10-28 (v0 deliverable)** and **11-04 (v1 baseline):** B3 on Haiku facets; the similarity-weighted error was
  29.99% on 373 training targets (`v1/notebooks/similarity_methods.ipynb`).
- **11-10:** the GPT judge filter (3.4) and source-scope sensitivity (`source_scope_sensitivity.py`,
  `split_peer_eval.py`: valuation checks of profile variants; kept in v1 as the record).
- **11-13:** agent-ranking valuation on 20 targets (`ranking_20targets_sonnet`, v1 record).